In [1]:
import pandas as pd
import numpy as np

ROOT = r"C:\Users\Acer\Documents\AuthentiScore"

FEATURES = [
    "followers",
    "following",
    "posts",
    "bio_length",
    "has_profile_picture",
    "is_private",
    "username_digit_ratio",
    "follower_following_ratio",
    "profile_completeness"
]

print("M16 Rule-Based Risk Engine")
print("Features:", FEATURES)

M16 Rule-Based Risk Engine
Features: ['followers', 'following', 'posts', 'bio_length', 'has_profile_picture', 'is_private', 'username_digit_ratio', 'follower_following_ratio', 'profile_completeness']


In [2]:
df = pd.read_csv(
    ROOT + r"\data\processed\instagram_features.csv"
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (696, 10)


,followers,following,posts,bio_length,has_profile_picture,is_private,username_digit_ratio,fake,follower_following_ratio,profile_completeness
0,1000,955,32,53,1,0,0.27,0,1.047120,1.0
1,2740,533,286,44,1,0,0.00,0,5.140713,1.0
2,159,98,13,0,1,1,0.10,0,1.622449,1.0
3,414,651,679,82,1,0,0.00,0,0.635945,1.0
4,151,126,6,0,1,1,0.00,0,1.198413,1.0


In [3]:
def calculate_profile_rule_risk(profile):

    risk_score = 0
    signals = []

    # 1. Low follower/following ratio
    if profile["following"] > 0:

        ratio = profile["followers"] / profile["following"]

        if ratio < 0.1:
            risk_score += 20

            signals.append({
                "rule": "low_follower_following_ratio",
                "weight": 20,
                "message": "Very low follower-to-following ratio"
            })

    # 2. High username digit ratio
    if profile["username_digit_ratio"] > 0.5:

        risk_score += 15

        signals.append({
            "rule": "high_username_digit_ratio",
            "weight": 15,
            "message": "Username contains a high proportion of digits"
        })

    # 3. Missing profile picture
    if profile["has_profile_picture"] == 0:

        risk_score += 20

        signals.append({
            "rule": "missing_profile_picture",
            "weight": 20,
            "message": "Profile picture is missing"
        })

    # 4. Low profile completeness
    if profile["profile_completeness"] < 0.6:

        risk_score += 15

        signals.append({
            "rule": "low_profile_completeness",
            "weight": 15,
            "message": "Profile has low completeness"
        })

    # 5. No posts
    if profile["posts"] == 0:

        risk_score += 10

        signals.append({
            "rule": "no_posts",
            "weight": 10,
            "message": "Profile has no posts"
        })

    # 6. Empty bio
    if profile["bio_length"] == 0:

        risk_score += 10

        signals.append({
            "rule": "empty_bio",
            "weight": 10,
            "message": "Profile has no biography"
        })

    risk_score = min(risk_score, 100)

    return {
        "profile_rule_risk": risk_score,
        "signals": signals
    }

In [4]:
sample_profile = df.iloc[0]

result = calculate_profile_rule_risk(sample_profile)

print("Profile Rule Risk:", result["profile_rule_risk"])
print("Signals:")

for signal in result["signals"]:
    print(signal)

Profile Rule Risk: 0
Signals:


In [5]:
for index in range(5):

    profile = df.iloc[index]

    result = calculate_profile_rule_risk(profile)

    print("\n" + "=" * 60)
    print("Profile:", index)
    print("Fake Label:", profile["fake"])
    print("Rule Risk:", result["profile_rule_risk"])
    print("Signals:", result["signals"])


Profile: 0
Fake Label: 0.0
Rule Risk: 0
Signals: []

Profile: 1
Fake Label: 0.0
Rule Risk: 0
Signals: []

Profile: 2
Fake Label: 0.0
Rule Risk: 10
Signals: [{'rule': 'empty_bio', 'weight': 10, 'message': 'Profile has no biography'}]

Profile: 3
Fake Label: 0.0
Rule Risk: 0
Signals: []

Profile: 4
Fake Label: 0.0
Rule Risk: 10
Signals: [{'rule': 'empty_bio', 'weight': 10, 'message': 'Profile has no biography'}]
